# FinanceBench: Evaluation Playground


<hr style="border-bottom:0.1px solid gray">

##### (1) API Requirements
Add the following API keys into your `.env` file:

```ruby
OPENAI_API_KEY = 'INSERT API KEY HERE'
ANTHROPIC_API_KEY = 'INSERT API KEY HERE'
REPLICATE_API_TOKEN = 'INSERT API KEY HERE'
```

##### (2) Required Folder Structure

```bash
|-- /
|    |-- data/
|    |      | -- financebench_open_source.jsonl
     |      | -- financebench_document_information.jsonl
|    |-- pdfs/
|           | -- <... provided filings as PDF documents ...>
|    |-- results/
|    |-- vectorstores/
|    |-- evaluation_playground.ipynb
```


<br>
<hr style="border-bottom:0.1px solid gray">

In [1]:
%%capture
%pip install pymupdf

In [2]:
import os
import sys
import json
import pickle
import datetime
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from dotenv import load_dotenv

load_dotenv()

#LangChain Stuff
from langchain.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
# from langchain.embeddings import OpenAIEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.chains import RetrievalQA

# LangChain Model Wrappers
from langchain.chat_models import ChatOpenAI
from langchain.chat_models import ChatAnthropic
from langchain.llms.replicate import Replicate

from typing import List, Tuple, Union, Optional, Callable
# # Model Providers
# import openai
# import anthropic
# import replicate
# import tiktoken


# # import ANTHROPIC TOKENIZER
# CLIENT = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
# anthropic_tokenizer = CLIENT.get_tokenizer()
# openai.api_key = os.environ['OPENAI_API_KEY']

pd.set_option('display.max_columns', None)  # Show all columns

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Load gemini-2.5-pro using langchain wrapper
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [4]:

llm.invoke("Sing a ballad of LangChain.")

AIMessage(content="(Verse 1)\nIn digital realms, where logic spun so grand,\nA giant mind, a language deep, did dwell,\nBut in its vastness, bound by training's hand,\nNo outer world, its knowledge could compel.\n\n(Verse 2)\nIt dreamed of facts, beyond its core's embrace,\nTo reach the web, or parse a database's plea,\nTo wield a tool, and find truth in its place,\nBut lacked the hands, to set its wisdom free.\n\n(Verse 3)\nThen from the code, a framework did arise,\nWith Python's grace, and purpose clear and bright,\nLangChain its name, beneath the digital skies,\nTo weave connections, banish lonely night.\n\n(Verse 4)\nIt forged the 'Chains,' a path for thought to flow,\nFrom prompt to parse, and back again with might,\nA sequence clear, where data learns to grow,\nAnd complex tasks, are broken into light.\n\n(Verse 5)\nThen 'Agents' came, with reason in their core,\nTo ponder choices, what next step to take,\nAnd 'Tools' they wielded, knocking at the door,\nOf search engines, or da

In [29]:
##############################################################################
# MODEL CONFIGS
##############################################################################
configs = [
            {"provider": "google",     "model_name":"gemini-2.5-flash-lite",    "eval_mode":"closedBook",        "temp":0.01,   "max_tokens":2048},
            {"provider": "google",     "model_name":"gemini-2.5-flash-lite",    "eval_mode":"inContext",          "temp":0.01,   "max_tokens":2048},
            # {"provider": "openai",     "model_name":"gpt-4o-2024-05-13",   "eval_mode":"singleStore",        "temp":0.01,   "max_tokens":2048},
            # {"provider": "openai",     "model_name":"gpt-4o-2024-05-13",   "eval_mode":"sharedStore",        "temp":0.01,   "max_tokens":2048},
            # {"provider": "openai",     "model_name":"gpt-4o-2024-05-13",   "eval_mode":"inContext",          "temp":0.01,   "max_tokens":2048},
            # {"provider": "openai",     "model_name":"gpt-4o-2024-05-13",   "eval_mode":"inContext_reverse",  "temp":0.01,   "max_tokens":2048},
            # {"provider": "openai",     "model_name":"gpt-4o-2024-05-13",   "eval_mode":"oracle",             "temp":0.01,   "max_tokens":2048},
            # {"provider": "openai",     "model_name":"gpt-4o-2024-05-13",   "eval_mode":"oracle_reverse",     "temp":0.01,   "max_tokens":2048},
            # {"provider": "openai",     "model_name":"gpt-4o-2024-05-13",   "eval_mode":"sharedStore",        "temp":0.01,   "max_tokens":2048},
            # {"provider": "openai",     "model_name":"gpt-4-1106-preview",  "eval_mode":"sharedStore",        "temp":0.01,   "max_tokens":2048},
            # {"provider": "openai",     "model_name":"gpt-4-1106-preview",  "eval_mode":"singleStore",        "temp":0.01,   "max_tokens":2048},
            # {"provider": "openai",     "model_name":"gpt-4-1106-preview",  "eval_mode":"inContext",          "temp":0.01,   "max_tokens":2048},
            # {"provider": "openai",     "model_name":"gpt-4-1106-preview",  "eval_mode":"closedBook",         "temp":0.01,   "max_tokens":2048},
            # {"provider": "anthropic",  "model_name":"claude-2",            "eval_mode":"inContext",          "temp":0.01,   "max_tokens":2048},
            # {"provider": "openai",     "model_name":"gpt-4-1106-preview",  "eval_mode":"oracle",             "temp":0.01,   "max_tokens":2048},
            # {"provider": "replicate",  "model_name":"llama2",              "eval_mode":"sharedStore",        "temp":0.01,   "max_tokens":2048},
            # {"provider": "replicate",  "model_name":"llama2",              "eval_mode":"singleStore",        "temp":0.01,   "max_tokens":2048},
            # {"provider": "openai",     "model_name":"gpt-4",               "eval_mode":"sharedStore",        "temp":0.01,   "max_tokens":2048},
            # {"provider": "openai",     "model_name":"gpt-4",               "eval_mode":"singleStore",        "temp":0.01,   "max_tokens":2048},
            # {"provider": "openai",     "model_name":"gpt-4",               "eval_mode":"closedBook",         "temp":0.01,   "max_tokens":2048},
            # {"provider": "openai",     "model_name":"gpt-4",               "eval_mode":"oracle",             "temp":0.01,   "max_tokens":2048},
            # {"provider": "openai",     "model_name":"gpt-4-1106-preview",  "eval_mode":"oracle_reverse",     "temp":0.01,   "max_tokens":2048},
            # {"provider": "anthropic",  "model_name":"claude-2",            "eval_mode":"oracle_reverse",     "temp":0.01,   "max_tokens":2048},
            # {"provider": "openai",     "model_name":"gpt-4-1106-preview",  "eval_mode":"inContext_reverse",  "temp":0.01,   "max_tokens":2048},
            # {"provider": "anthropic",  "model_name":"claude-2",            "eval_mode":"inContext_reverse",  "temp":0.01,   "max_tokens":2048},
            {"provider": "",           "model_name":"",                    "eval_mode":"singleStore",        "temp":None,   "max_tokens":None},       # SPECIAL MODE --> RETRIEVAL ONLY MODE (SINGLE STORE)
            {"provider": "",           "model_name":"",                    "eval_mode":"sharedStore",        "temp":None,   "max_tokens":None},       # SPECIAL MODE --> RETRIEVAL ONLY MODE (SHARED STORE)
]

replicate_model_mapping = dict({
            "llama2": "meta/llama-2-70b-chat:02e509c789964a7ea8736978a43525956ef40397be9033abf9fd2badfe68c9e3"
        })

##############################################################################
# DATASET CONFIG
##############################################################################
PATH_CURRENT = os.path.abspath(os.getcwd())
PATH_DATASET_JSONL = PATH_CURRENT + "/data/financebench_open_source.jsonl"
PATH_DOCUMENT_INFO_JSONL = PATH_CURRENT + "/data/financebench_document_information.jsonl"
PATH_RESULTS = PATH_CURRENT + "/results/"
PATH_PDFS = PATH_CURRENT + "/pdfs/"

# Choose DATASET PORTION:
# - ALL: Full Dataset
# - OPEN_SOURCE: Open Source Part (n=150)
# - CLOSED_SOURCE: Closed Source Part --> Request access at contact@patronus.ai
DATASET_PORTION = "OPEN_SOURCE"   

##############################################################################
# VECTOR STORE SETUP
##############################################################################
VS_CHUNK_SIZE = 1024
VS_CHUNK_OVERLAP = 30
VS_DIR_VS = PATH_CURRENT + "/vectorstores"

In [15]:
##############################################################################
# LOAD DATASET
##############################################################################

# Load Full Dataset 
df_questions = pd.read_json(PATH_DATASET_JSONL, lines=True)
df_meta = pd.read_json(PATH_DOCUMENT_INFO_JSONL, lines=True)
df_full = pd.merge(df_questions, df_meta, on="doc_name")

# Get all docs
df_questions = df_questions.sort_values('doc_name')
ALL_DOCS = df_questions['doc_name'].unique().tolist()
print(f"Total number of distinct PDF: {len(ALL_DOCS)}")

# Select relevant dataset portion
if DATASET_PORTION != "ALL":
    df_questions = df_questions.loc[df_questions["dataset_subset_label"]==DATASET_PORTION]
print(f"Number of questions: {len(df_questions)}")

# Check relevant documents
df_questions = df_questions.sort_values('doc_name')
docs = df_questions['doc_name'].unique().tolist()
print(f"Number of distinct PDF: {len(docs)}")

Total number of distinct PDF: 84
Number of questions: 150
Number of distinct PDF: 84


In [16]:
df_questions["question"].head(20)

0     What is the FY2018 capital expenditure amount ...
1     Assume that you are a public equities analyst....
2     Is 3M a capital-intensive business based on FY...
3     What drove operating margin change as of FY202...
4     If we exclude the impact of M&A, which segment...
5     Does 3M have a reasonably healthy liquidity pr...
6     Which debt securities are registered to trade ...
7     Does 3M maintain a stable trend of dividend di...
8     What is the FY2019 fixed asset turnover ratio ...
9     What is the FY2017 - FY2019 3 year average of ...
10    You are an investment banker and your only res...
11    What is Adobe's year-over-year change in unadj...
12    What is the FY2017 operating cash flow ratio f...
13    Does Adobe have an improving operating margin ...
14    Does Adobe have an improving Free cashflow con...
15    What is the quantity of restructuring costs di...
16    Roughly how many times has AES Corporation sol...
17    Based on the information provided primaril

In [17]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
vector = embeddings.embed_query("hello, world!")
vector[:5]

[0.05636945366859436,
 0.004828543867915869,
 -0.07625909894704819,
 -0.023642510175704956,
 0.053293220698833466]

In [37]:

##############################################################################
# HELPER FUNCTIONS (PDF-PARSING + VECTOR-STORE SETUPS)
##############################################################################
def get_pdf_text(doc: str) -> List:
    """Extracts text from a PDF file using PyMuPDFLoader.
    Args:
        doc (str): The name of the PDF file (without .pdf extension) located in PATH_PDFS.
    Returns:
        list: A list of document objects containing the extracted text from the PDF.
    Notes:
        - Assumes the PDF file exists in the directory specified by PATH_PDFS.
        - Uses PyMuPDFLoader from LangChain for efficient PDF text extraction.
    """
    path_doc = f"{PATH_PDFS}/{doc}.pdf"
    pdf_reader = PyMuPDFLoader(path_doc)
    pdf_text = pdf_reader.load()

    return pdf_text

def build_vectorstore_retriever(docs: Union[str, List[str]], 
    embeddings: Callable = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")) -> Tuple:
    """Creates or loads a Chroma vector store and returns a retriever for querying it.
    Args:
        docs: Either the string 'all' (to process all documents in ALL_DOCS) or a single document name (without .pdf extension).
        embeddings: Embedding function to convert text to vectors. Defaults to GoogleGenerativeAIEmbeddings(model='gemini-embedding-001models/text-embedding-004').
    Returns:
        A tuple containing:
            - retriever: A retriever object for querying the vector store.
            - vectordb: The Chroma vector database instance.
    Notes:
        - If the vector store doesn't exist, it creates a new one by extracting text from the specified PDFs, splitting into chunks, embedding, and persisting to disk.
        - Uses RecursiveCharacterTextSplitter with VS_CHUNK_SIZE and VS_CHUNK_OVERLAP for text chunking.
        - Persists the database to a directory (VS_DIR_VS/shared for 'all', or VS_DIR_VS/doc_name for a single doc).
        - If the database exists, it loads it instead of recreating it.
        - TODO: Investigate the impact of chunk size on vectorization and model performance.
    """

    if docs == "all":
        docs = ALL_DOCS
        db_path = VS_DIR_VS + "/shared"
    else:
        docs = [docs]
        db_path = VS_DIR_VS + "/" + docs[0]
    
    # Create Vector Store if not already existing
    if not os.path.exists(db_path):
        
        # Create folder for vector store
        os.mkdir(db_path) 

        # Create vector store itself --> chrom.sqlite3 database
        if not os.path.exists(f"{db_path}/chroma.sqlite3"):
            vectordb = Chroma(persist_directory=db_path, embedding_function=embeddings)
            vectordb.persist() # saves the vector database in disk for referencing in the future
    
            # Add Documents to Vector store    
            for doc in docs:
                pdf_text = get_pdf_text(doc)
                text_splitter = RecursiveCharacterTextSplitter(
                    chunk_size = VS_CHUNK_SIZE,
                    chunk_overlap = VS_CHUNK_OVERLAP,
                )
                splitted_texts = text_splitter.split_documents(pdf_text)
                # TODO: check if the chunk size would affect the vectorisation process and therefore the model results
        
                # Add to vector store
                vectordb.add_documents(documents=splitted_texts)
                vectordb.persist()

    else:
        # reload previously saved chroma vector db
        vectordb = Chroma(persist_directory=db_path, embedding_function=embeddings)

    # return a retriever object and the database instance
    return vectordb.as_retriever(), vectordb

##############################################################################
# MODEL + CALL HANDLERS
##############################################################################
def get_max_context_length(prompt: str, gemini_cutoff: int = 122576, model_name: str = "gemini-1.5-pro-001") -> int:
    """Calculates the maximum number of characters to fit within tokenizer limits for Gemini models.
    Args:
        prompt: The input text to be tokenized.
        gemini_cutoff: Maximum token limit for Gemini models. Defaults to 1,048,576 (suitable for Gemini 1.5 Flash/Pro).
        model_name: The Gemini model name for the tokenizer (e.g., 'gemini-1.5-pro-001'). Defaults to 'gemini-1.5-pro-001'.
    Returns:
        The number of characters that fit within the Gemini model's token limit.
    Notes:
        - Uses Vertex AI SDK's local tokenizer (requires google-cloud-aiplatform[tokenization] >= 1.57.0).
        - The tokenizer vocabulary is downloaded and cached locally on first use, enabling offline operation.
        - If the prompt exceeds the token limit, uses binary search to find the largest prefix <= cutoff tokens.
        - Original Anthropic and OpenAI code is commented out as they are not used.
    """
    from vertexai.preview import tokenization
    
    # Initialize Gemini tokenizer
    tokenizer = tokenization.get_tokenizer_for_model(model_name)
    
    # Check Gemini Tokenizer
    nb_tokens_gemini = tokenizer.count_tokens(prompt).total_tokens
    number_of_chars_gemini = len(prompt)
    
    if nb_tokens_gemini > gemini_cutoff:
        # Binary search to find the maximum character length where token count <= cutoff
        low = 0
        high = len(prompt)
        while low < high:
            mid = (low + high + 1) // 2
            prefix = prompt[:mid]
            if tokenizer.count_tokens(prefix).total_tokens <= gemini_cutoff:
                low = mid
            else:
                high = mid - 1
        number_of_chars_gemini = low

    """
    # (0) Check Anthropic Tokenizer
    tokens_anthropic = anthropic_tokenizer.encode(prompt)
    nb_tokens_anthropic = len(tokens_anthropic)
    number_of_chars_anthropic = len(prompt)
    
    if nb_tokens_anthropic > anthropic_cutoff:
        tokens_anthropic_tokens = tokens_anthropic.tokens
        token_lengths_anthropic = [len(token) for token in tokens_anthropic_tokens]
        number_of_chars_anthropic = sum(token_lengths_anthropic[:anthropic_cutoff])

    # (1) Check OpenAI Tokenizer
    tokenizer_openai = tiktoken.encoding_for_model("gpt-4-1106-preview")
    tokens_openai = tokenizer_openai.encode(prompt)
    nb_tokens_openai = len(tokens_openai)
    number_of_chars_openai = len(prompt)

    if nb_tokens_openai > openai_cutoff:
        tokens_openai_tokens = [tokenizer_openai.decode_single_token_bytes(token) for token in tokens_openai]
        token_lengths_openai = [len(token) for token in tokens_openai_tokens]
        number_of_chars_openai = sum(token_lengths_openai[:openai_cutoff])

    # Cut prompt depending on minimal length limit
    number_of_chars = min(number_of_chars_openai, number_of_chars_anthropic)
    """

    return number_of_chars_gemini

def get_model(provider: str = "google", model_name: str = "gemini-2.5-flash", temp: float = 0.01, max_tokens: int = 2048) -> \
    Optional[Union[ChatOpenAI, ChatAnthropic, Replicate, ChatGoogleGenerativeAI]]:
    """Initializes a language model based on the specified provider and parameters.
    Args:
        provider: The model provider ('openai', 'anthropic', 'replicate', or 'google')
        model_name: The specific model name (e.g., 'gpt-4' for OpenAI).
        temp: Temperature for controlling randomness in model output
        max_tokens: Maximum number of tokens to generate
    Returns:
        An initialized model instance (e.g., ChatOpenAI, ChatAnthropic, Replicate, ChatGoogleGenerativeAI) or None if provider is invalid.
    Raises:
        ValueError: If an unknown model is specified for the 'replicate' provider.
    Notes:
        - Requires environment variable ANTHROPIC_API_KEY for Anthropic models.
        - Uses replicate_model_mapping for Replicate provider model names.
    """
    if provider == "google":
        return ChatGoogleGenerativeAI(
            model=model_name, 
            temperature=temp, 
            max_output_tokens=max_tokens
            )
        
    # elif provider == "anthropic":
    #     return ChatAnthropic(
    #         model=model_name,
    #         temperature=temp, 
    #         max_tokens_to_sample=max_tokens, 
    #         anthropic_api_key=os.environ['ANTHROPIC_API_KEY']
    #         )
    
    # elif provider == "replicate":
    #     if model_name in replicate_model_mapping:
    #         return Replicate(
    #             model=replicate_model_mapping[model_name],
    #             model_kwargs={
    #                 'temperature': temp, 
    #                 'max_new_tokens': max_tokens
    #                 },
    #         )
    #     else:
    #         raise ValueError("Unknown Model")
    
    # elif provider == "openai":
    #     return ChatOpenAI(
    #         model_name=model_name, 
    #         temperature=temp, 
    #         max_tokens=max_tokens
    #         )
        
    else:
        return None


def get_answer(model: Optional[Union[ChatOpenAI, ChatAnthropic, Replicate, ChatGoogleGenerativeAI]], 
    eval_mode: str, question: str, context: str, retriever: object, retriever_only: bool = False) -> Tuple[str, List]:
    """Generates an answer to a question using a specified model, evaluation mode, and context or retriever.
    Args:
        model: The language model instance (e.g., ChatOpenAI, ChatAnthropic) or None for retriever-only mode.
        eval_mode: Evaluation mode ('closedBook', 'oracle', 'oracle_reverse', 'inContext', 'inContext_reverse', 
        'singleStore', or 'sharedStore').
        question: The question to answer.
        context: The context or evidence text to use in certain modes.
        retriever: A retriever object for querying a vector store (used in 'singleStore' or 'sharedStore' modes).
        retriever_only: If True, returns only retrieved documents without invoking the model. Defaults to False.
    Returns:
        A tuple containing:
            - answer: The generated answer (empty string if retriever_only=True).
            - retrieved_documents: List of retrieved documents (empty for non-retrieval modes).
    Notes:
        - 'closedBook': Answers without any context, relying on model knowledge. Only for reference
        - 'oracle'/'oracle_reverse': Uses "cheating" context. Only for reference. 
        - 'inContext'/'inContext_reverse': Uses context of max size = get_max_context_length(). Assumes that 
        the important context is only in the early tokens for big documents with more than 100,000 tokens. 
        - 'singleStore'/'sharedStore': RAG method that uses retriever from vector db for context. 
        - In retriever-only mode, only the retriever is queried, and no answer is generated. For reference only. 
    """
    retrieved_documents = []

    if eval_mode == "closedBook":
        prompt = f"Answer this question: {question}"
        answer = model.predict(prompt)
        
    elif eval_mode == "oracle":
        prompt = f"Answer this question: {question} \nHere is the relevant evidence that you need to answer the question:\n[START OF FILING] {context} [END OF FILING]"
        answer = model.predict(prompt)

    elif eval_mode == "oracle_reverse":
        
        prompt = f"Context:\n[START OF FILING] {context} [END OF FILING\n\n Answer this question: {question} \n"
        answer = model.predict(prompt)

    elif eval_mode in ["inContext",  "inContext_reverse"]:
        
        # Context Cutoff to satisfy max tokens
        max_number_of_chars = get_max_context_length(context)
        context = context[:max_number_of_chars]
        
        if eval_mode == "inContext":
            prompt = f"Answer this question: {question} \nHere is the relevant filing that you need to answer the question:\n[START OF FILING] {context} [END OF FILING]"
        else:
            prompt = f"Context:\n[START OF FILING] {context} [END OF FILING]\n\n Answer this question: {question}\n"

        answer = model.predict(prompt)

    elif eval_mode == "singleStore" or eval_mode == "sharedStore":
        
        # Retrieval-only mode if model=None (No LLM calls, only queries in VectorDB)
        if not model:           
            prompt = f"{question}"
            s = retriever.invoke(prompt)
            return ("", s)

        else:

            # Don't add a question prefix as RetrievalQA will do some automatic prompt wrapping
            # --> This can replace by more advanced Retrieval Strategies
            prompt = f"{question}"
            qa = RetrievalQA.from_chain_type(
                llm=model,
                chain_type="stuff",
                retriever=retriever,
                return_source_documents=True,
            )
            s = qa(prompt)
            
            answer = s["result"]
            retrieved_documents = s["source_documents"]


    
    return (answer, retrieved_documents)


In [38]:
##############################################################################
# EVALUATION
##############################################################################
import time

# Specify evaluation model
model_config = configs[1]

# Set evaluation questions
df_eval = df_questions


# Get the model
model = get_model(provider=model_config["provider"],
                  model_name=model_config["model_name"],
                  temp=model_config["temp"],
                  max_tokens=model_config["max_tokens"])

print(f"--> Evaluating: {model_config['model_name']} / {model_config['eval_mode']}")

last_docs = None
results = []

# Run evaluation on the model  --> Sort along doc_name to reuse retriever configs in memory
for k, (idx, row) in tqdm(enumerate(df_eval.sort_values("doc_name").head(50).iterrows()), total=len(df_eval)):
    if k % 5 == 0:
        print(f"---------entry {k} out of {len(df_eval)} entries ----------------")
    
    # (A) Setup Context or Retriever
    if model_config["eval_mode"] == "closedBook":
        retriever = None
        context = ""
    
    elif model_config["eval_mode"] in ["inContext", "inContext_reverse"]:
        retriever = None
        docs = row["doc_name"]
        if not (last_docs == docs):
            pages = get_pdf_text(row["doc_name"])
            context = "\n\n".join([page.page_content for page in pages])
            
    
    elif model_config["eval_mode"] in ["oracle", "oracle_reverse"]:
        context = "\n\n".join([evidence["evidence_text_full_page"] for evidence in row["evidence"]])
        retriever = None

    elif model_config["eval_mode"] in ["singleStore", "sharedStore"]:
        context = ""
        docs = "all"

        if model_config["eval_mode"] == "singleStore":
            docs = row["doc_name"]
        
        if not (last_docs == docs):
            retriever, _ = build_vectorstore_retriever(docs=docs)
            last_docs = docs


    else:
        raise ValueError("Unknown 'eval_mode'!")


    # (B) Model Call
    (answer, retrieved_documents) = get_answer(
                                        model=model, 
                                        eval_mode=model_config["eval_mode"], 
                                        question=row["question"], 
                                        context=context, 
                                        retriever=retriever
                                        )
    

    # (C) Bookkeeping
    results.append({
                        **model_config, 
                        "financebench_id" : row["financebench_id"],
                        "question" : row["question"],
                        "gold_answer": row["answer"],
                        "model_answer": answer,
                        "retrieved_documents" : retrieved_documents,
                    })
    time.sleep(20)

df_results = pd.DataFrame(results)
df_results.to_csv(PATH_RESULTS + "/" + model_config["model_name"] + "_" + model_config["eval_mode"] + ".csv")

--> Evaluating: gemini-2.5-flash-lite / inContext


  0%|          | 0/150 [00:00<?, ?it/s]

---------entry 0 out of 150 entries ----------------
---------entry 5 out of 150 entries ----------------
---------entry 10 out of 150 entries ----------------


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token_count"
  quota_id: "GenerateContentInputTokensPerModelPerMinute-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash-lite"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 250000
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 45
}
].
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 4.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check you

---------entry 15 out of 150 entries ----------------


...

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token_count"
  quota_id: "GenerateContentInputTokensPerModelPerMinute-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash-lite"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 250000
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 48
}
].
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 4.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check you

---------entry 40 out of 150 entries ----------------


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token_count"
  quota_id: "GenerateContentInputTokensPerModelPerMinute-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash-lite"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 250000
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 52
}
].
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 4.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check you

---------entry 45 out of 150 entries ----------------


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token_count"
  quota_id: "GenerateContentInputTokensPerModelPerMinute-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash-lite"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 250000
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 5
}
].
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 4.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your

In [ ]:
import gradio as gr
# Gradio interface
def process_question(question: str, doc_name: str, eval_mode: str, context: str = ""):
    # Initialize model
    model = get_model()
    
    # Setup context or retriever based on eval_mode
    retriever = None
    if eval_mode in ["inContext", "inContext_reverse"]:
        if doc_name and doc_name != "all":
            pages = get_pdf_text(doc_name)
            context = "\n\n".join([page.page_content for page in pages])
        else:
            context = ""
    elif eval_mode in ["singleStore", "sharedStore"]:
        docs = "all" if eval_mode == "sharedStore" else doc_name
        if docs:
            retriever, _ = build_vectorstore_retriever(docs=docs)
    
    # Get answer and retrieved documents
    answer, retrieved_docs = get_answer(
        model=model,
        eval_mode=eval_mode,
        question=question,
        context=context,
        retriever=retriever
    )
    
    # Format output
    output = f"**Answer:**\n{answer}\n\n"
    if retrieved_docs:
        output += "**Retrieved Documents:**\n"
        for i, doc in enumerate(retrieved_docs, 1):
            output += f"**Document {i}:**\n{doc.page_content[:500]}...\n\n"
    
    return output

# Create Gradio interface
with gr.Blocks() as demo:
    gr.Markdown("# FinDocGPT")
    
    with gr.Row():
        question_input = gr.Textbox(label="Enter your question", placeholder="Type your question here...")
        doc_dropdown = gr.Dropdown(choices=["all"] + ALL_DOCS, label="Select Document", value="all")
        eval_mode_dropdown = gr.Dropdown(
            choices=["closedBook", "oracle", "inContext"],
            label="Context Type",
            value="sharedStore"
        )
    
    context_input = gr.Textbox(
        label="Additional context (optional)",
        placeholder="Enter context here",
        lines=5
    )
    
    submit_button = gr.Button("Submit")
    output = gr.Markdown(label="Answer and Retrieved Documents")
    
    submit_button.click(
        fn=process_question,
        inputs=[question_input, doc_dropdown, eval_mode_dropdown, context_input],
        outputs=output
    )

# Launch the interface
demo.launch()

/home/dsfee222/miniconda3/envs/financebench/lib/python3.13/site-packages/gradio/components/dropdown.py:230: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: sharedStore or set allow_custom_value=True.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
